# Client API cho slide chọn thị trường: World Bank, phân trang, retry, lỗi trong HTTP 200

**Bối cảnh.** Một công ty thương mại điện tử Việt Nam tính mở thị trường thứ hai trong ASEAN. Head of Expansion cần
một bảng cho slide: dân số, GDP bình quân đầu người, tỉ lệ dân dùng internet của 10 nước ASEAN giai đoạn 2015–2023,
và **dân số online** (dân số × tỉ lệ dùng internet) để ước quy mô thị trường. Số phải lấy lại được khi World Bank cập
nhật, nên bạn viết một client nhỏ thay vì tải tay CSV.

Nguồn: [World Bank Indicators API v2](https://datahelpdesk.worldbank.org/knowledgebase/articles/889392) (miễn phí, không
cần key, dữ liệu CC BY 4.0). Ba chỉ số: `SP.POP.TOTL` (dân số), `NY.GDP.PCAP.CD` (GDP/người, USD hiện hành),
`IT.NET.USER.ZS` (% dân số dùng internet). Response là **mảng hai phần tử**: `[metadata, bản_ghi]`, metadata có
`page`, `pages`, `per_page`, `total`, `lastupdated`.

Ba cái bẫy của API này mà client phải xử lý:

1. **Phân trang**: mặc định 50 bản ghi một trang; chỉ đọc trang 1 thì thiếu dữ liệu mà không có lỗi nào.
2. **Lỗi tạm thời**: 429 (quá nhiều request) và 5xx thì thử lại có backoff; 4xx khác thì báo ngay.
3. **Lỗi nằm trong HTTP 200**: tham số sai vẫn trả `200` với body `[{"message": [...]}]`.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` gọi API thật (11 request); khi chấm chúng bị bỏ qua, test dùng server giả.
- Chấm: `uv run pytest tracks/data-analyst/modules/05-data-collection/exercises/intermediate/01-world-bank-api-client -q`

In [ ]:
import json
import time
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

import httpx
import numpy as np
import pandas as pd

## 1. Một request đáng tin: `get_json`

`get_json(client, path, params, max_retries=3, sleep=time.sleep)` gọi `client.get(path, params=params)` (client là
`httpx.Client` có `base_url`) và trả JSON đã parse:

- lỗi mạng (`httpx.TransportError`), HTTP 429 và 5xx: thử lại tối đa `max_retries` lần; trước lần thử thứ `k`
  (k = 0, 1, 2…) gọi `sleep(2 ** k)`, riêng khi response có header `Retry-After` thì nghỉ đúng số giây đó;
- hết lượt thử: `ApiError`;
- HTTP 4xx khác: `ApiError` ngay, không thử lại (gửi lại request sai vẫn sai);
- HTTP 200 nhưng body là mảng mà phần tử đầu có khoá `"message"`: `ApiError` kèm nội dung `message`, không thử lại.

In [ ]:
RETRY_STATUS = {429, 500, 502, 503, 504}


class ApiError(RuntimeError):
    """API trả lỗi không thử lại được, hoặc đã hết lượt thử."""


def get_json(client: httpx.Client, path: str, params: dict, max_retries: int = 3, sleep=time.sleep):
    """GET có retry cho lỗi tạm thời; báo lỗi rõ cho lỗi 4xx và lỗi nằm trong body."""
    for attempt in range(max_retries + 1):
        last = attempt == max_retries
        try:
            resp = client.get(path, params=params, timeout=30)
        except httpx.TransportError as exc:
            if last:
                raise ApiError(f"Lỗi mạng sau {attempt + 1} lần thử: {exc}") from exc
            sleep(2 ** attempt)
            continue
        if resp.status_code in RETRY_STATUS:
            if last:
                raise ApiError(f"HTTP {resp.status_code} sau {attempt + 1} lần thử")
            retry_after = resp.headers.get("Retry-After")
            sleep(int(retry_after) if retry_after else 2 ** attempt)
            continue
        if resp.status_code >= 400:
            raise ApiError(f"HTTP {resp.status_code}: {resp.text[:200]}")
        body = resp.json()
        if isinstance(body, list) and body and isinstance(body[0], dict) and "message" in body[0]:
            raise ApiError(f"API báo lỗi trong body: {body[0]['message']}")
        return body

In [ ]:
# explore: lỗi nằm trong HTTP 200 - gọi một chỉ số không tồn tại
client = httpx.Client(base_url="https://api.worldbank.org/v2", headers={"User-Agent": "da-learning-course/1.0"})
raw = client.get("/country/VNM/indicator/NOT.AN.INDICATOR", params={"format": "json"})
print(raw.status_code, raw.text[:160])
try:
    get_json(client, "/country/VNM/indicator/NOT.AN.INDICATOR", {"format": "json"})
except ApiError as exc:
    print("ApiError:", exc)

## 2. Lấy đủ mọi trang: `fetch_indicator`

`fetch_indicator(client, indicator, countries, start, end, per_page=1000, sleep=time.sleep) -> list[dict]`:

- path `/country/<mã1;mã2;...>/indicator/<indicator>`, tham số `format=json`, `date="<start>:<end>"`, `per_page`,
  `page` (bắt đầu từ 1); mọi request đi qua `get_json` (truyền `sleep` xuống);
- đọc `pages` trong metadata, gọi lần lượt tới trang cuối, nối mọi bản ghi theo thứ tự;
- không có dữ liệu thì API trả `[metadata, None]` với `pages = 0`: trả `[]`;
- cuối cùng so số bản ghi nhận được với `total` trong metadata; lệch thì `ApiError`.

In [ ]:
def fetch_indicator(client: httpx.Client, indicator: str, countries: list[str], start: int, end: int,
                    per_page: int = 1000, sleep=time.sleep) -> list[dict]:
    """Mọi bản ghi của một chỉ số cho các nước và khoảng năm, đi hết các trang."""
    path = f"/country/{';'.join(countries)}/indicator/{indicator}"
    params = {"format": "json", "date": f"{start}:{end}", "per_page": per_page}
    records, page = [], 1
    while True:
        meta, data = get_json(client, path, {**params, "page": page}, sleep=sleep)
        records.extend(data or [])
        if page >= int(meta["pages"]):
            break
        page += 1
    if len(records) != int(meta["total"]):
        raise ApiError(f"Nhận {len(records)} bản ghi, metadata báo {meta['total']}")
    return records

## 3. Từ JSON tới bảng cho slide

`to_tidy(records) -> pd.DataFrame`: mỗi bản ghi một dòng, cột `iso3` (`countryiso3code`), `country`
(`country.value`), `indicator` (`indicator.id`), `year` (số nguyên từ `date`), `value` (số thực, giữ `NaN` khi API trả
`null`). Bỏ bản ghi có `iso3` rỗng (nhóm nước như "East Asia & Pacific"). Sắp theo `iso3, indicator, year`, index
0..n-1.

`online_population(tidy, year) -> pd.DataFrame`: chỉ lấy năm `year`, mỗi nước một dòng, index `iso3`, cột theo thứ
tự `country, population, internet_pct, gdp_per_capita, online_population` (tên cột từ `INDICATORS`); `online_population
= population * internet_pct / 100`, giữ `NaN` nếu thiếu tỉ lệ internet (đừng biến thành 0). Sắp giảm dần theo
`online_population`, `NaN` xuống cuối.

In [ ]:
INDICATORS = {"SP.POP.TOTL": "population", "IT.NET.USER.ZS": "internet_pct", "NY.GDP.PCAP.CD": "gdp_per_capita"}


def to_tidy(records: list[dict]) -> pd.DataFrame:
    """iso3, country, indicator, year, value; bỏ nhóm nước; sắp iso3, indicator, year."""
    rows = [{
        "iso3": r["countryiso3code"],
        "country": r["country"]["value"],
        "indicator": r["indicator"]["id"],
        "year": int(r["date"]),
        "value": np.nan if r["value"] is None else float(r["value"]),
    } for r in records if r["countryiso3code"]]
    out = pd.DataFrame(rows, columns=["iso3", "country", "indicator", "year", "value"])
    out["year"] = out["year"].astype("int64")
    out["value"] = out["value"].astype("float64")
    return out.sort_values(["iso3", "indicator", "year"]).reset_index(drop=True)


def online_population(tidy: pd.DataFrame, year: int) -> pd.DataFrame:
    """Bảng một dòng mỗi nước cho năm year, kèm dân số online."""
    one = tidy[(tidy["year"] == year) & tidy["indicator"].isin(INDICATORS)]
    wide = one.pivot_table(index="iso3", columns="indicator", values="value", aggfunc="first", dropna=False)
    wide = wide.rename(columns=INDICATORS).reindex(columns=list(INDICATORS.values()))
    wide.insert(0, "country", one.groupby("iso3")["country"].first())
    wide["online_population"] = wide["population"] * wide["internet_pct"] / 100
    wide.columns.name = None
    out = wide[["country", "population", "internet_pct", "gdp_per_capita", "online_population"]]
    return out.sort_values("online_population", ascending=False, na_position="last")

## 4. Không để lộ bí mật trong log

World Bank không cần key, nhưng phần lớn API trả phí có (`?api_key=...`). URL đầy đủ hay bị in ra log, lưu vào metadata
hoặc dán vào ticket. `redact(url, secrets=("api_key", "key", "token")) -> str`: thay giá trị của các tham số query có tên
trong `secrets` bằng `***`, giữ nguyên mọi phần khác (thứ tự tham số, path).

In [ ]:
def redact(url: str, secrets: tuple[str, ...] = ("api_key", "key", "token")) -> str:
    """URL với giá trị các tham số bí mật thay bằng ***."""
    parts = urlsplit(url)
    query = [(k, "***" if k.lower() in secrets else v) for k, v in parse_qsl(parts.query, keep_blank_values=True)]
    return urlunsplit(parts._replace(query=urlencode(query, safe="*")))

In [ ]:
# explore: gọi thật 3 chỉ số cho 10 nước ASEAN, lưu JSON thô kèm metadata trước khi parse
from datetime import datetime, timezone
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data/raw_api/world-bank-asean"
RAW.mkdir(parents=True, exist_ok=True)
ASEAN = ["BRN", "KHM", "IDN", "LAO", "MYS", "MMR", "PHL", "SGP", "THA", "VNM"]

records = []
for indicator in INDICATORS:
    data = fetch_indicator(client, indicator, ASEAN, 2015, 2023, per_page=50)
    meta = client.get(f"/country/{';'.join(ASEAN)}/indicator/{indicator}",
                      params={"format": "json", "date": "2015:2023", "per_page": 1}).json()[0]
    (RAW / f"{indicator}.json").write_text(json.dumps({
        "fetched_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "url": redact(f"{client.base_url}/country/{';'.join(ASEAN)}/indicator/{indicator}?format=json&date=2015:2023"),
        "lastupdated": meta["lastupdated"],
        "total": meta["total"],
        "records": data,
    }, ensure_ascii=False))
    print(f"{indicator}: {len(data)} bản ghi, lastupdated {meta['lastupdated']}")
    records.extend(data)

tidy = to_tidy(records)
print(tidy.shape, "giá trị thiếu:", int(tidy.value.isna().sum()))
print(tidy[tidy.value.isna()])

In [ ]:
# explore: bảng cho slide năm 2023 và thay đổi so với 2015
table = online_population(tidy, 2023)
print((table.assign(population=lambda d: d.population / 1e6, online_population=lambda d: d.online_population / 1e6)
       .round(1).rename(columns={"population": "population_m", "online_population": "online_m"})))
growth = tidy.pivot_table(index=["iso3", "year"], columns="indicator", values="value")
print((growth.xs(2023, level="year") / growth.xs(2015, level="year")).round(2)
      .rename(columns=INDICATORS)[["gdp_per_capita", "internet_pct"]])

In [ ]:
# explore: biểu đồ cho slide
import matplotlib.pyplot as plt

plot = table.dropna(subset=["online_population"]).sort_values("online_population")
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(plot.country, plot.online_population / 1e6, color="#4c78a8")
ax.set_xlabel("triệu người dùng internet (2023)")
ax.set_title("Indonesia, Philippines, Việt Nam: ba thị trường online lớn nhất ASEAN", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 5. Đọc kết quả, rồi làm trắc nghiệm

1. Với `per_page=50`, mỗi chỉ số cần bao nhiêu request cho 10 nước × 9 năm? Nếu `per_page=1000` thì sao? Khi nào
   nên giảm `per_page`?
2. Nước nào thiếu tỉ lệ internet những năm gần đây? Trên slide bạn ghi thế nào (đừng để ô trống thành "0")?
3. Ba thị trường online lớn nhất năm 2023 là gì? GDP/người khác nhau ra sao, và vì sao "dân số online" chưa đủ để
   chọn thị trường?
4. Vì sao lưu `lastupdated` và thời điểm tải cạnh dữ liệu thô?

Xong thì mở app, làm phần **trắc nghiệm** của bài.